In [0]:
%run ../Includes/common_functions

In [0]:
%run ../Includes/config


In [0]:
v_data_source = dbutils.widgets.get("p_data_source")
v_file_date = dbutils.widgets.get("p_file_date")
print(v_data_source)
print(v_file_date)
print(raw_folder_path)

In [0]:

from pyspark.sql.functions import col

constructors_schema = "constructorId STRING, constructorRef STRING, name STRING, nationality STRING, url STRING"

constructor_df = spark.read \
.schema(constructors_schema) \
.format("json") \
.load(f"{raw_folder_path}/{v_file_date}/constructors.json")

constructor_dropped_df = constructor_df.drop(col('url'))

In [0]:

from pyspark.sql.functions import current_timestamp, lit
constructor_final_df = constructor_dropped_df.withColumnRenamed("constructorId", "constructor_id") \
                                             .withColumnRenamed("constructorRef", "constructor_ref") \
                                             .withColumn("ingestion_date", current_timestamp())\
                                             .withColumn("data_source", lit(v_data_source)) \
                                             .withColumn("file_date", lit(v_file_date))

(constructor_final_df.write.mode("overwrite")
 .format("delta")
 .option("mergeSchema", "true")
 .save("abfss://dev@f1storage02.dfs.core.windows.net/Bronze/constructors")
)

